# PestCLEF 2026 - Knowledge Graph Extraction v6
**Strategy**: Mistral 7B (prompt abstract v5) + **post-processing + dictionary augmentation**

**Texte reale**: EPOP corpus (Dataverse) - 247 documente, verificate (99.6% offset match)

**v6 Îmbunătățiri**:
1. Post-processing: normalizare la forme canonice cunoscute
2. Dictionary augmentation: entități cunoscute scanate în textul test
3. Deduplicare inteligentă Mistral + dictionary

**Metric**: Macro-averaged F1

## Section 1 - Install Dependencies

In [ ]:
%%capture
!pip install gliner transformers accelerate bitsandbytes sentencepiece
print('Dependencies installed')

## Section 2 - Load Data & Utilities

In [ ]:
import json, csv, re, os, zipfile, requests, time
from collections import defaultdict, Counter
from pathlib import Path
import torch

DATA_DIR = Path('/kaggle/input/competitions/pest-clef-2026')
with open(DATA_DIR / 'PestCLEF-2026_train.json') as f: train = json.load(f)
with open(DATA_DIR / 'PestCLEF-2026_dev.json') as f:   dev   = json.load(f)
with open(DATA_DIR / 'PestCLEF-2026_test.json') as f:  test  = json.load(f)
print(f'Train: {len(train)} | Dev: {len(dev)} | Test: {len(test)}')

VALID_PREDICATES = {'Affects','Causes','Dispersed_by','Found_on','Located_in','Occurs_on','Transmits'}
SCHEMA_LOOKUP = defaultdict(set)
for s,p,o in [
    ('Disease','Affects','Plant'),('Pest','Causes','Disease'),
    ('Disease','Dispersed_by','Dissemination_pathway'),('Pest','Dispersed_by','Dissemination_pathway'),
    ('Pest','Found_on','Plant'),('Pest','Found_on','Dissemination_pathway'),
    ('Vector','Found_on','Plant'),('Vector','Found_on','Dissemination_pathway'),
    ('Disease','Located_in','Location'),('Pest','Located_in','Location'),
    ('Plant','Located_in','Location'),('Vector','Located_in','Location'),
    ('Disease','Occurs_on','Date'),('Pest','Occurs_on','Date'),
    ('Plant','Occurs_on','Date'),('Vector','Occurs_on','Date'),
    ('Vector','Transmits','Disease'),('Vector','Transmits','Pest'),
]:
    SCHEMA_LOOKUP[(s,o)].add(p)

print('Schema definită')

In [ ]:
# Descarcă textele reale EPOP
TEXT_DIR = '/kaggle/working/texts'
os.makedirs(TEXT_DIR, exist_ok=True)

zip_path = '/kaggle/working/EPOP_documents.zip'
if not os.path.exists(zip_path):
    print('Descarcă EPOP_documents.zip...')
    resp = requests.get(
        'https://entrepot.recherche.data.gouv.fr/api/access/datafile/719158',
        timeout=120)
    with open(zip_path, 'wb') as f: f.write(resp.content)
    print(f'  OK: {len(resp.content)/1024:.0f} KB')

BASE_DIR = f'{TEXT_DIR}/EPOP_documents'
if not os.path.exists(BASE_DIR):
    with zipfile.ZipFile(zip_path) as z: z.extractall(TEXT_DIR)

doc_texts = {}
for split in ['train','dev','test']:
    d = f'{BASE_DIR}/{split}'
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.endswith('.txt'):
                doc_texts[f[:-4]] = f'{d}/{f}'

print(f'Texte disponibile: {len(doc_texts)}')
print(f'Test coverage: {sum(1 for d in test if d["doc_id"] in doc_texts)}/{len(test)}')

def get_doc_text(doc):
    did = doc['doc_id']
    if did in doc_texts:
        with open(doc_texts[did], encoding='utf-8') as f:
            return f.read()
    if 'text_bound_annotations' in doc:
        ents = sorted(doc['text_bound_annotations']['entities'],
                      key=lambda e: e['offsets'][0][0])
        return ' | '.join(f"{e['type']}: {e['form']}" for e in ents)
    return ''

print(f'Sample: {get_doc_text(test[0])[:150]}')

In [ ]:
# Entity vocabulary
ENTITY_VOCAB = {}
ENTITY_FORMS = defaultdict(set)
KNOWN_FORMS  = set()  # toate formele acceptate din KG gold

for doc in train + dev:
    for ent in doc['text_bound_annotations']['entities']:
        ENTITY_VOCAB[ent['form'].lower()] = ent['type']
        ENTITY_FORMS[ent['type']].add(ent['form'])
    for e in doc['knowledge_graph']:
        for f in e['subject']: KNOWN_FORMS.add(f.lower())
        for f in e['object']:  KNOWN_FORMS.add(f.lower())

# Map lowercase -> original form (pentru normalizare)
FORM_CANONICAL = {}
for doc in train + dev:
    for e in doc['knowledge_graph']:
        for f in e['subject']: FORM_CANONICAL[f.lower()] = f
        for f in e['object']:  FORM_CANONICAL[f.lower()] = f

print(f'Entity vocab: {len(ENTITY_VOCAB)} | Known forms: {len(KNOWN_FORMS)}')

# Evaluation functions
def normalize(s):
    return s.lower().replace('_','').replace(' ','')

def edge_matches(pred_edge, ref_edge):
    if normalize(pred_edge['predicate']) != normalize(ref_edge['predicate']):
        return False
    return (pred_edge['subject'].lower() in [s.lower() for s in ref_edge['subject']] and
            pred_edge['object'].lower()  in [o.lower() for o in ref_edge['object']])

def f1_for_doc(preds, refs):
    T, P = len(refs), len(preds)
    if T == 0 and P == 0: return 1.0
    if T == 0 or P == 0:  return 0.0
    tp = sum(1 for pe in preds if any(edge_matches(pe,re) for re in refs))
    prec, rec = tp/P, tp/T
    return 2*prec*rec/(prec+rec) if prec+rec > 0 else 0.0

def evaluate(predictions, dataset, verbose=False):
    scores = []
    for doc in dataset:
        s = f1_for_doc(predictions.get(doc['doc_id'],[]), doc['knowledge_graph'])
        scores.append(s)
        if verbose and s < 0.3:
            print(f'  Low F1={s:.2f} doc {doc["doc_id"]}: '
                  f'{len(predictions.get(doc["doc_id"],[]))} pred vs {len(doc["knowledge_graph"])} gold')
    macro = sum(scores)/len(scores)
    print(f'  Macro-F1: {macro:.4f} | docs: {len(scores)} | '
          f'avg_pred: {sum(len(predictions.get(d["doc_id"],[])) for d in dataset)/len(dataset):.1f} '
          f'avg_gold: {sum(len(d["knowledge_graph"]) for d in dataset)/len(dataset):.1f}')
    return macro

# LOCAL SCORING FUNCTION
def local_score(predictions, dataset=dev, label=''):
    """
    Calculează F1 local pe dev set — folosește ÎNAINTE de a submite pe Kaggle.
    predictions: dict {doc_id -> [edges]}
    """
    scores = []
    precisions, recalls = [], []
    for doc in dataset:
        pred = predictions.get(doc['doc_id'], [])
        ref  = doc['knowledge_graph']
        T, P = len(ref), len(pred)
        if T == 0 and P == 0:
            scores.append(1.0); precisions.append(1.0); recalls.append(1.0)
            continue
        if T == 0 or P == 0:
            scores.append(0.0); precisions.append(0.0 if P>0 else 1.0); recalls.append(0.0)
            continue
        tp = sum(1 for pe in pred if any(edge_matches(pe,re) for re in ref))
        prec, rec = tp/P, tp/T
        f1 = 2*prec*rec/(prec+rec) if prec+rec > 0 else 0.0
        scores.append(f1); precisions.append(prec); recalls.append(rec)
    macro_f1   = sum(scores)/len(scores)
    avg_prec   = sum(precisions)/len(precisions)
    avg_rec    = sum(recalls)/len(recalls)
    print(f'\nLOCAL SCORE {"["+label+"]" if label else ""}')
    print(f'  Macro-F1:  {macro_f1:.4f}')
    print(f'  Precision: {avg_prec:.4f}')
    print(f'  Recall:    {avg_rec:.4f}')
    print(f'  Docs:      {len(scores)}')
    print(f'  Avg pred edges: {sum(len(predictions.get(d["doc_id"],[])) for d in dataset)/len(dataset):.1f}')
    print(f'  Avg gold edges: {sum(len(d["knowledge_graph"]) for d in dataset)/len(dataset):.1f}')
    # Best and worst docs
    doc_scores = [(doc['doc_id'], s) for doc, s in zip(dataset, scores)]
    doc_scores.sort(key=lambda x: x[1])
    print(f'  Worst 3 docs: {doc_scores[:3]}')
    print(f'  Best  3 docs: {doc_scores[-3:]}')
    return macro_f1

# Submission Generator
def make_submission(predictions, output_path='/kaggle/working/submission.csv'):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(['doc_id','knowledge_graph'])
        for doc in test:
            edges = predictions.get(doc['doc_id'], [])
            seen, unique = set(), []
            for e in edges:
                k = (e['predicate'], e['subject'].lower(), e['object'].lower())
                if k not in seen: seen.add(k); unique.append(e)
            writer.writerow([doc['doc_id'], json.dumps(unique, ensure_ascii=False)])
    print(f'Submission: {output_path}')

print('Toate utilitarele gata')

In [ ]:
pred_freq = Counter()
for doc in train + dev:
    for e in doc['knowledge_graph']:
        pred_freq[e['predicate']] += 1
print('Predicate frequencies:', dict(pred_freq.most_common()))

## System C: Mistral 7B - Prompt Abstract (v5)
**Modificări v5 vs v4:**
- Prompt ABSTRACT fără exemple cu entități reale (care biasează modelul)
- Context 2000 chars (în loc de 4000)
- 3 few-shot examples (în loc de 5)
- 512 max new tokens (în loc de 800)
- Aceste setări au dat **0.24 F1** în versiunea v3

In [ ]:
import gc, ctypes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def free_vram(label=''):
    torch.cuda.empty_cache(); gc.collect()
    try: ctypes.CDLL('libc.so.6').malloc_trim(0)
    except: pass
    if torch.cuda.is_available():
        free = (torch.cuda.get_device_properties(0).total_memory
                - torch.cuda.memory_allocated()) / 1e9
        print(f'  VRAM liberă{" ("+label+")" if label else ""}: {free:.1f} GB')

free_vram('before load')
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
mistral_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
mistral_tokenizer.pad_token = mistral_tokenizer.eos_token
mistral_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto')
mistral_model.eval()
free_vram('after load')
print('Mistral 7B încărcat')

In [ ]:
# Prompt v5 (abstract — a dat 0.24)
SYSTEM_PROMPT = """You are an expert in plant pathology and knowledge graph extraction.
Extract a knowledge graph from the given text. Output ONLY a valid JSON array of triples.
Each triple must have exactly three fields: predicate, subject, object.

Schema:
- Pest Causes Disease
- Disease Affects Plant
- Vector Transmits Disease or Pest
- Pest/Vector Found_on Plant or Dissemination_pathway
- Disease/Pest/Plant/Vector Located_in Location
- Disease/Pest/Plant/Vector Occurs_on Date
- Disease/Pest Dispersed_by Dissemination_pathway

Rules:
1. Output ONLY the JSON array, no explanation, no markdown fences.
2. Use exact entity names from the text.
3. Only use these predicates: Affects, Causes, Dispersed_by, Found_on, Located_in, Occurs_on, Transmits
4. Be comprehensive."""

def build_few_shot_examples(n=3):
    examples = []
    used_preds = set()
    for doc in sorted(train, key=lambda d: len(d['knowledge_graph']), reverse=True):
        if len(examples) >= n: break
        preds = {e['predicate'] for e in doc['knowledge_graph']}
        if preds - used_preds:
            text = get_doc_text(doc)[:800]
            kg_list = []
            for e in doc['knowledge_graph'][:15]:
                s = e['subject'][0] if isinstance(e['subject'],list) else e['subject']
                o = e['object'][0]  if isinstance(e['object'], list) else e['object']
                kg_list.append({'predicate':e['predicate'],'subject':s,'object':o})
            examples.append({'text': text, 'kg': json.dumps(kg_list, ensure_ascii=False)})
            used_preds |= preds
    print(f'Few-shot: {len(examples)} exemple, predicate: {used_preds}')
    return examples

FEW_SHOT = build_few_shot_examples(n=3)

def build_prompt(doc_text, few_shot):
    messages = [
        {'role': 'user',      'content': SYSTEM_PROMPT + '\n\nReady? Reply OK.'},
        {'role': 'assistant', 'content': 'OK'},
    ]
    for ex in few_shot:
        messages.append({'role': 'user',      'content': f"Text: {ex['text']}"})
        messages.append({'role': 'assistant', 'content': ex['kg']})
    messages.append({'role': 'user', 'content': f"Text: {doc_text[:2000]}"})
    return messages

def mistral_generate(messages, max_new_tokens=512):
    encoded = mistral_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=False)
    if isinstance(encoded, dict):
        input_ids = encoded['input_ids'].to(mistral_model.device)
        attn_mask = encoded.get('attention_mask')
        if attn_mask is not None: attn_mask = attn_mask.to(mistral_model.device)
    else:
        input_ids = encoded.to(mistral_model.device)
        attn_mask = None
    with torch.inference_mode():
        out = mistral_model.generate(
            input_ids, attention_mask=attn_mask,
            max_new_tokens=max_new_tokens, do_sample=False,
            temperature=1.0, pad_token_id=mistral_tokenizer.eos_token_id)
    return mistral_tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)

def parse_output(raw):
    edges = []
    raw = re.sub(r'```json|```', '', raw).strip()
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    if not match: return edges
    try: parsed = json.loads(match.group())
    except: return edges
    for item in parsed:
        if not isinstance(item, dict): continue
        pred = item.get('predicate','').strip()
        subj = str(item.get('subject','')).strip()
        obj  = str(item.get('object','')).strip()
        if not (pred and subj and obj): continue
        matched_pred = None
        for vp in VALID_PREDICATES:
            if normalize(pred) == normalize(vp):
                matched_pred = vp; break
        if not matched_pred: continue
        edges.append({'predicate': matched_pred, 'subject': subj, 'object': obj})
    return edges

# ÎMBUNĂTĂȚIRE v6 #1: Post-processing - normalizare la forme canonice

# Build map: pentru fiecare formă cunoscută din KG gold, lista tuturor alias-urilor
FORM_GROUPS = []  # list of sets of equivalent forms
for doc in train + dev:
    for e in doc['knowledge_graph']:
        FORM_GROUPS.append(set(f.lower() for f in e['subject']))
        FORM_GROUPS.append(set(f.lower() for f in e['object']))

# Flatten: lowercase_form -> group_id
FORM_TO_GROUP = {}
for i, group in enumerate(FORM_GROUPS):
    for form in group:
        if form not in FORM_TO_GROUP:
            FORM_TO_GROUP[form] = i

# Known entities from TBA: form -> (canonical_form, type)
KNOWN_ENTITIES = {}  # lowercase_form -> (canonical, type)
for doc in train + dev:
    for ent in doc['text_bound_annotations']['entities']:
        form_lower = ent['form'].lower()
        if form_lower not in KNOWN_ENTITIES:
            KNOWN_ENTITIES[form_lower] = (ent['form'], ent['type'])

print(f'KNOWN_ENTITIES: {len(KNOWN_ENTITIES)} forme')

def normalize_form(form):
    """Încearcă să mapeze forma la cel mai apropiat match cunoscut."""
    if not form:
        return form
    form_lower = form.lower().strip()
    # Exact match
    if form_lower in KNOWN_ENTITIES:
        return KNOWN_ENTITIES[form_lower][0]
    # Try removing trailing 's' (plural)
    if form_lower.endswith('s') and form_lower[:-1] in KNOWN_ENTITIES:
        return KNOWN_ENTITIES[form_lower[:-1]][0]
    # Try substring match — caută un alias cunoscut în form
    for known_lower, (canonical, _) in KNOWN_ENTITIES.items():
        if len(known_lower) >= 4 and known_lower == form_lower:
            return canonical
    # Nu găsim — returnăm forma originală
    return form

# ÎMBUNĂTĂȚIRE v6 #2: Dictionary augmentation - entități cunoscute în text

def find_entities_in_text(text):
    """Găsește toate entitățile cunoscute care apar literal în text."""
    text_lower = text.lower()
    found = []
    for form_lower, (canonical, etype) in KNOWN_ENTITIES.items():
        if len(form_lower) < 3:  # prea scurt, va da false positives
            continue
        if form_lower in text_lower:
            found.append((canonical, etype))
    # Deduplicate
    seen, unique = set(), []
    for c, t in found:
        if c.lower() not in seen:
            seen.add(c.lower())
            unique.append((c, t))
    return unique

def dictionary_augment(text, max_edges_per_pair=1):
    """
    Pentru fiecare pereche de entități găsite în text care formează o pereche
    schema-validă, emit triplete cu predicatul cel mai frecvent.
    """
    entities = find_entities_in_text(text)
    edges = []
    seen = set()
    for form_i, type_i in entities:
        for form_j, type_j in entities:
            if form_i == form_j: continue
            valid_preds = SCHEMA_LOOKUP.get((type_i, type_j), set())
            if not valid_preds: continue
            best_pred = max(valid_preds, key=lambda p: pred_freq.get(p, 0))
            key = (form_i.lower(), best_pred, form_j.lower())
            if key not in seen:
                seen.add(key)
                edges.append({'predicate': best_pred, 'subject': form_i, 'object': form_j})
    return edges

# SYSTEM C v6: Mistral + post-processing + dictionary augmentation

def system_c_v6(doc, use_dict_aug=True):
    """Mistral + normalization + optional dictionary augmentation."""
    text = get_doc_text(doc)
    if not text: return []
    
    # Step 1: Mistral extraction
    messages = build_prompt(text, FEW_SHOT)
    raw = mistral_generate(messages)
    mistral_edges = parse_output(raw)
    
    # Step 2: Post-processing — normalize forms
    normalized = []
    for e in mistral_edges:
        normalized.append({
            'predicate': e['predicate'],
            'subject':   normalize_form(e['subject']),
            'object':    normalize_form(e['object']),
        })
    
    # Step 3: Dictionary augmentation (optional)
    if use_dict_aug:
        dict_edges = dictionary_augment(text)
        # Combine: Mistral first (higher precision), then dict edges
        seen = set()
        combined = []
        for e in normalized + dict_edges:
            key = (e['predicate'], e['subject'].lower(), e['object'].lower())
            if key not in seen:
                seen.add(key)
                combined.append(e)
        return combined
    
    return normalized

print('System C v6 definit — cu post-processing și dictionary augmentation')

In [ ]:
CACHE_DIR = '/kaggle/working/cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# Dev: testăm 3 variante pentru comparație
print('=' * 60)
print('RULĂM MISTRAL PE DEV (o singură dată, cache rezultatele)')
print('=' * 60)

preds_mistral_raw_dev = {}  # Mistral raw (fără post-processing)
t0 = time.time()
for i, doc in enumerate(dev):
    text = get_doc_text(doc)
    if not text:
        preds_mistral_raw_dev[doc['doc_id']] = []
        continue
    messages = build_prompt(text, FEW_SHOT)
    raw = mistral_generate(messages)
    preds_mistral_raw_dev[doc['doc_id']] = parse_output(raw)
    elapsed = time.time() - t0
    eta = elapsed/(i+1)*(len(dev)-i-1)
    print(f'  [{i+1}/{len(dev)}] {doc["doc_id"]} '
          f'({len(preds_mistral_raw_dev[doc["doc_id"]])} edges) — ETA:{eta:.0f}s')

# Test VARIANT A: Mistral raw
print('\n--- VARIANT A: Mistral RAW ---')
score_a = local_score(preds_mistral_raw_dev, dev, label='A: Mistral raw')

# Test VARIANT B: Mistral + normalize
print('\n--- VARIANT B: Mistral + normalize forms ---')
preds_norm_dev = {}
for doc_id, edges in preds_mistral_raw_dev.items():
    preds_norm_dev[doc_id] = [{
        'predicate': e['predicate'],
        'subject':   normalize_form(e['subject']),
        'object':    normalize_form(e['object']),
    } for e in edges]
score_b = local_score(preds_norm_dev, dev, label='B: Mistral + normalize')

# Test VARIANT C: Mistral + normalize + dictionary
print('\n--- VARIANT C: Mistral + normalize + dictionary augmentation ---')
preds_full_dev = {}
for doc in dev:
    text = get_doc_text(doc)
    norm_edges = preds_norm_dev.get(doc['doc_id'], [])
    dict_edges = dictionary_augment(text) if text else []
    seen, combined = set(), []
    for e in norm_edges + dict_edges:
        key = (e['predicate'], e['subject'].lower(), e['object'].lower())
        if key not in seen:
            seen.add(key)
            combined.append(e)
    preds_full_dev[doc['doc_id']] = combined
score_c = local_score(preds_full_dev, dev, label='C: Full pipeline')

print('\n' + '=' * 60)
print('SUMAR PE DEV:')
print(f'  A (Mistral raw):                    {score_a:.4f}')
print(f'  B (Mistral + normalize):            {score_b:.4f}')
print(f'  C (Mistral + normalize + dict aug): {score_c:.4f}')
print('=' * 60)

# Alege cea mai bună variantă
best = max([('A', score_a), ('B', score_b), ('C', score_c)], key=lambda x: x[1])
print(f'\n CEL MAI BUN PE DEV: Varianta {best[0]} cu F1={best[1]:.4f}')

# Save all
with open(f'{CACHE_DIR}/preds_mistral_raw_dev.json','w') as f: json.dump(preds_mistral_raw_dev,f)
with open(f'{CACHE_DIR}/preds_norm_dev.json','w') as f: json.dump(preds_norm_dev,f)
with open(f'{CACHE_DIR}/preds_full_dev.json','w') as f: json.dump(preds_full_dev,f)

In [ ]:
# Test set: rulăm Mistral o dată, apoi generăm toate cele 3 variante 
print('=' * 60)
print('RULĂM MISTRAL PE TEST')
print('=' * 60)

preds_mistral_raw_test = {}
t0 = time.time()
for i, doc in enumerate(test):
    text = get_doc_text(doc)
    if not text:
        preds_mistral_raw_test[doc['doc_id']] = []
        continue
    messages = build_prompt(text, FEW_SHOT)
    raw = mistral_generate(messages)
    preds_mistral_raw_test[doc['doc_id']] = parse_output(raw)
    elapsed = time.time() - t0
    eta = elapsed/(i+1)*(len(test)-i-1)
    print(f'  [{i+1}/{len(test)}] {doc["doc_id"]} '
          f'({len(preds_mistral_raw_test[doc["doc_id"]])} edges) — ETA:{eta:.0f}s')

# Generate variants
preds_norm_test = {}
for doc_id, edges in preds_mistral_raw_test.items():
    preds_norm_test[doc_id] = [{
        'predicate': e['predicate'],
        'subject':   normalize_form(e['subject']),
        'object':    normalize_form(e['object']),
    } for e in edges]

preds_full_test = {}
for doc in test:
    text = get_doc_text(doc)
    norm_edges = preds_norm_test.get(doc['doc_id'], [])
    dict_edges = dictionary_augment(text) if text else []
    seen, combined = set(), []
    for e in norm_edges + dict_edges:
        key = (e['predicate'], e['subject'].lower(), e['object'].lower())
        if key not in seen:
            seen.add(key)
            combined.append(e)
    preds_full_test[doc['doc_id']] = combined

# Save toate 3 submission-urile
make_submission(preds_mistral_raw_test, '/kaggle/working/submission_v6_A_raw.csv')
make_submission(preds_norm_test,        '/kaggle/working/submission_v6_B_normalized.csv')
make_submission(preds_full_test,        '/kaggle/working/submission_v6_C_full.csv')

# Default submission = cea mai bună pe dev
make_submission(preds_full_test, '/kaggle/working/submission.csv')

print('\n Submission-uri generate:')
print('  submission_v6_A_raw.csv        — Mistral raw')
print('  submission_v6_B_normalized.csv — Mistral + normalize')
print('  submission_v6_C_full.csv       — Mistral + normalize + dict')
print('  submission.csv                 — cea mai bună (default = C)')
print('\n Submite toate 3 pe Kaggle și vezi care scorează cel mai bine!')